### ЗАДАЧА: Пакетная загрузка конфигов деплоя

От DevOps-команды приходит пакет строк с конфигами сервисов для выкладки.
Нужно обработать их так, чтобы:
- валидные конфиги попали в итоговый список,
- проблемные записи не остановили весь пакет,
- по ошибкам собрался отдельный журнал,
- в конце было видно, какие сервисы включены по окружениям и какой у них средний timeout.

Часть строк содержит ошибки в формате и числах,
часть использует неизвестное окружение или неправильный флаг включения.


In [26]:
# service|max_retries|timeout_sec|environment|enabled
rows = [
    'auth|3|1.5|prod|on',
    'billing|0|2.0|stage|on',
    'search|two|0.8|dev|off',
    'media|5|-1|prod|on',
    'chat|4|1.2|test|off',
    'mail|2|0.5|stage|maybe',
    'worker|1|3.4|prod|on',
]


class DeployConfigError(Exception):
    pass


class RowFormatError(DeployConfigError):
    pass


class RetriesError(DeployConfigError):
    pass


class TimeoutError(DeployConfigError):
    pass


class EnvironmentError(DeployConfigError):
    pass


class EnabledFlagError(DeployConfigError):
    pass


def parse_config(row):
    # TODO: распарсить строку и провалидировать max_retries, timeout_sec, environment, enabled
    parts = row.split("|")
    if len(parts) != 5:
        raise RowFormatError("Строка должна состоять из 5 символов")
    
    service, max_retries, timeout_sec, environment, enabled = parts
    # TODO: при ошибках конвертации использовать raise ... from ...
    try:
        max_retries = int(max_retries)
    except ValueError as e:
        raise RetriesError("Попытки должны быть числом")
    
    if max_retries < 0:
        raise RetriesError("Попытки не должны быть отрицательными")
    
    try:
        timeout_sec = float(timeout_sec)
    except ValueError as e:
        raise TimeoutError("Время должно быть числом")
    
    if timeout_sec < 0:
        raise TimeoutError("Время не должно быть отрицательным")
    
    allowed_environment = {"prod", "stage", "dev"}
    if environment not in allowed_environment:
        raise EnvironmentError("Недопустимая среда")
        
    # TODO: enabled вернуть как bool
    if enabled == "on":
        enabled = True
    else:
        enabled = False
        raise EnabledFlagError("Должно быть включено")
    
    return {
        "service": service,
        "max_retries": max_retries,
        "timeout_sec": timeout_sec,
        "environment": environment,
        "enabled": enabled
    }


def load_configs(rows):
    # TODO: вернуть (configs, errors)
    configs = []
    errors = []
    for row in rows:
        try:
            configs.append(parse_config(row))
        except DeployConfigError as e:
            errors.append((row, type(e).__name__, e))
    return configs, errors
  

# TODO: вызвать load_configs(rows)
configs, errors = load_configs(rows)
# TODO: вывести число валидных конфигов и число ошибок
print(f"Число валидных конфигов: {len(configs)}")
for config in configs:
    print(config)
print(f"Число ошибок: {len(errors)}")
# TODO: вывести ошибки по типам
errors_by_type = {}
for name, error, message in errors:
    errors_by_type[error] = errors_by_type.get(error, 0) + 1
    print(f"Ошибка: '{error}', сообщение: '{message}', строка: '{name}'")
for error, count in errors_by_type.items():
    print(f"Ошибка '{error}' встречается {count} раз.")
# TODO: собрать enabled_by_environment: dict[str, list[str]]
enabled_by_environment = {}
for config in configs:
    enabled_by_environment.setdefault(config["environment"], [])
    enabled_by_environment[config["environment"]] = [*enabled_by_environment[config["environment"]], config]
for a, b in enabled_by_environment.items():
    print(a, b)
# TODO: посчитать average_timeout только по enabled=True
average_timeout = 0
for config in configs:
        average_timeout += config["timeout_sec"]
average_timeout = average_timeout / len(configs)
print(f"Среднее значение по enabled=True = {average_timeout} сек.")

Число валидных конфигов: 3
{'service': 'auth', 'max_retries': 3, 'timeout_sec': 1.5, 'environment': 'prod', 'enabled': True}
{'service': 'billing', 'max_retries': 0, 'timeout_sec': 2.0, 'environment': 'stage', 'enabled': True}
{'service': 'worker', 'max_retries': 1, 'timeout_sec': 3.4, 'environment': 'prod', 'enabled': True}
Число ошибок: 4
Ошибка: 'RetriesError', сообщение: 'Попытки должны быть числом', строка: 'search|two|0.8|dev|off'
Ошибка: 'TimeoutError', сообщение: 'Время не должно быть отрицательным', строка: 'media|5|-1|prod|on'
Ошибка: 'EnvironmentError', сообщение: 'Недопустимая среда', строка: 'chat|4|1.2|test|off'
Ошибка: 'EnabledFlagError', сообщение: 'Должно быть включено', строка: 'mail|2|0.5|stage|maybe'
Ошибка 'RetriesError' встречается 1 раз.
Ошибка 'TimeoutError' встречается 1 раз.
Ошибка 'EnvironmentError' встречается 1 раз.
Ошибка 'EnabledFlagError' встречается 1 раз.
prod [{'service': 'auth', 'max_retries': 3, 'timeout_sec': 1.5, 'environment': 'prod', 'enabled': 